# LAB 3: LÀM SẠCH DỮ LIỆU CƠ BẢN
**Dataset:** `patient_heart_rate.csv` – Dữ liệu y khoa về huyết áp bệnh nhân

**Mục tiêu:** Sử dụng Pandas để xử lý 8 vấn đề phổ biến trong dữ liệu thô:
1. Thiếu dòng tiêu đề
2. Nhiều biến lưu ở một cột
3. Đơn vị không nhất quán
4. Dòng trống
5. Dòng trùng lặp
6. Ký tự non-ASCII
7. Giá trị bị mất (Missing values)
8. Tiêu đề cột là giá trị chứ không phải tên biến

##  Bước 0 – Import thư viện & Upload dữ liệu

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print(' Import thư viện thành công!')

✅ Import thư viện thành công!


In [6]:
# Upload file patient_heart_rate.csv lên Colab
from google.colab import files
uploaded = files.upload()   # Chọn file patient_heart_rate.csv từ máy tính

Saving patient_heart_rate.csv to patient_heart_rate (1).csv


---
## Vấn đề 1: Thiếu dòng tiêu đề trong file CSV (Missing header)

In [8]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Thêm header vào dataframe để diễn giải dữ liệu
column_names = ["Id", "Name", "Age", "Weight",
                 'm0006', 'm0612', 'm1218', 'f0006', 'f0612', 'f1218']

# Đọc file dữ liệu (không có header sẵn → cung cấp names thủ công)
# Sử dụng encoding='utf-8-sig' để xử lý BOM và on_bad_lines='skip' để bỏ qua các dòng bị lỗi
df = pd.read_csv("patient_heart_rate.csv", names=column_names, encoding='utf-8-sig', on_bad_lines='skip')

# Hiển thị một vài dòng dữ liệu đầu tiên ra màn hình
print('=== DỮ LIỆU SAU KHI THÊM HEADER ===')
print(df.head(10))
print(f'\nShape: {df.shape}')

=== DỮ LIỆU SAU KHI THÊM HEADER ===
    Id            Name   Age      Weight m0006 m0612 m1218 f0006 f0612 f1218
0  1.0    Mickéy Mousé  56.0       70kgs    72    69    71     -     -     -
1  2.0     Donald Duck  34.0   154.89lbs     -     -     -    85    84    76
2  3.0      Mini Mouse  16.0         NaN     -     -     -    65    69    72
3  4.0  Scrooge McDuck   NaN       78kgs    78    79    72     -     -     -
4  5.0    Pink Panther  54.0  198.658lbs     -     -     -    69   NaN    75
5  6.0     Huey McDuck  52.0      189lbs     -     -     -    68    75    72
6  7.0    Dewey McDuck  19.0       56kgs     -     -     -    71    78    75
7  8.0      Scööpy Doo  32.0       78kgs    78    76    75     -     -     -
8  NaN             NaN   NaN         NaN   NaN   NaN   NaN   NaN   NaN   NaN
9  NaN             NaN   NaN         NaN   NaN   NaN   NaN   NaN   NaN   NaN

Shape: (16, 10)


---
## Vấn đề 2: Một cột lưu hỗn hợp nhiều dữ liệu
Cột `Name` chứa cả `Firstname` và `Lastname` → cần tách thành 2 cột riêng

In [9]:
# Tách cột Name thành Firstname và Lastname
df[['Firstname', 'Lastname']] = df['Name'].str.split(expand=True)
df = df.drop('Name', axis=1)

print('=== DỮ LIỆU SAU KHI TÁCH CỘT NAME ===')
print(df.head(10))

=== DỮ LIỆU SAU KHI TÁCH CỘT NAME ===
    Id   Age      Weight m0006 m0612 m1218 f0006 f0612 f1218 Firstname  \
0  1.0  56.0       70kgs    72    69    71     -     -     -    Mickéy   
1  2.0  34.0   154.89lbs     -     -     -    85    84    76    Donald   
2  3.0  16.0         NaN     -     -     -    65    69    72      Mini   
3  4.0   NaN       78kgs    78    79    72     -     -     -   Scrooge   
4  5.0  54.0  198.658lbs     -     -     -    69   NaN    75      Pink   
5  6.0  52.0      189lbs     -     -     -    68    75    72      Huey   
6  7.0  19.0       56kgs     -     -     -    71    78    75     Dewey   
7  8.0  32.0       78kgs    78    76    75     -     -     -    Scööpy   
8  NaN   NaN         NaN   NaN   NaN   NaN   NaN   NaN   NaN       NaN   
9  NaN   NaN         NaN   NaN   NaN   NaN   NaN   NaN   NaN       NaN   

  Lastname  
0    Mousé  
1     Duck  
2    Mouse  
3   McDuck  
4  Panther  
5   McDuck  
6   McDuck  
7      Doo  
8      NaN  
9      NaN  


---
## Vấn đề 3: Đơn vị đo lường không nhất quán (cột Weight)
Một số giá trị có đơn vị `lbs` lẫn với `kg` → cần chuyển hết về `kg`

In [10]:
# Lấy cột Weight để xử lý
weight = df['Weight']

for i in range(0, len(weight)):
    x = str(weight[i])

    # Nếu đơn vị là lbs thì loại bỏ và chuyển đổi
    if "lbs" in x:
        # Loại bỏ đơn vị lbs khỏi giá trị
        x = x[:-3]
        # Chuyển string sang float
        float_x = float(x)
        # Chuyển đổi sang kg và lưu dạng int
        y = int(float_x / 2.2)
        # Chuyển lại thành string có đơn vị kgs
        y = str(y) + "kgs"
        weight[i] = y

df['Weight'] = weight
print('=== DỮ LIỆU SAU KHI CHUẨN HÓA ĐƠN VỊ WEIGHT (kg) ===')
print(df.head(10))

=== DỮ LIỆU SAU KHI CHUẨN HÓA ĐƠN VỊ WEIGHT (kg) ===
    Id   Age Weight m0006 m0612 m1218 f0006 f0612 f1218 Firstname Lastname
0  1.0  56.0  70kgs    72    69    71     -     -     -    Mickéy    Mousé
1  2.0  34.0  70kgs     -     -     -    85    84    76    Donald     Duck
2  3.0  16.0    NaN     -     -     -    65    69    72      Mini    Mouse
3  4.0   NaN  78kgs    78    79    72     -     -     -   Scrooge   McDuck
4  5.0  54.0  90kgs     -     -     -    69   NaN    75      Pink  Panther
5  6.0  52.0  85kgs     -     -     -    68    75    72      Huey   McDuck
6  7.0  19.0  56kgs     -     -     -    71    78    75     Dewey   McDuck
7  8.0  32.0  78kgs    78    76    75     -     -     -    Scööpy      Doo
8  NaN   NaN    NaN   NaN   NaN   NaN   NaN   NaN   NaN       NaN      NaN
9  NaN   NaN    NaN   NaN   NaN   NaN   NaN   NaN   NaN       NaN      NaN


---
## Vấn đề 4: Dòng dữ liệu rỗng (toàn bộ giá trị là NaN)

In [11]:
print(f'Số dòng trước khi xóa dòng rỗng: {len(df)}')

# Xóa các dòng mà TẤT CẢ giá trị đều là NaN
df.dropna(how="all", inplace=True)

print(f'Số dòng sau khi xóa dòng rỗng : {len(df)}')
print('\n=== DỮ LIỆU SAU KHI XÓA DÒNG RỖNG ===')
print(df)

Số dòng trước khi xóa dòng rỗng: 16
Số dòng sau khi xóa dòng rỗng : 14

=== DỮ LIỆU SAU KHI XÓA DÒNG RỖNG ===
      Id   Age Weight m0006 m0612 m1218 f0006 f0612 f1218 Firstname Lastname
0    1.0  56.0  70kgs    72    69    71     -     -     -    Mickéy    Mousé
1    2.0  34.0  70kgs     -     -     -    85    84    76    Donald     Duck
2    3.0  16.0    NaN     -     -     -    65    69    72      Mini    Mouse
3    4.0   NaN  78kgs    78    79    72     -     -     -   Scrooge   McDuck
4    5.0  54.0  90kgs     -     -     -    69   NaN    75      Pink  Panther
5    6.0  52.0  85kgs     -     -     -    68    75    72      Huey   McDuck
6    7.0  19.0  56kgs     -     -     -    71    78    75     Dewey   McDuck
7    8.0  32.0  78kgs    78    76    75     -     -     -    Scööpy      Doo
10   9.0  52.0  85kgs     -     -     -    68    75    72      Huey   McDuck
11  10.0  12.0  45kgs     -     -     -    92    95    87     Louie   McDuck
12  11.0   NaN  60kgs    78    75    72    

---
## Vấn đề 5: Dữ liệu bị trùng lặp hoàn toàn
Trùng theo các cột `Firstname`, `Lastname`, `Age`, `Weight` → chỉ giữ lại 1 dòng

In [12]:
print(f'Số dòng trước khi loại trùng lặp: {len(df)}')

# Chỉ giữ lại 1 dòng cho mỗi nhóm dữ liệu trùng lặp hoàn toàn
df = df.drop_duplicates(subset=['Firstname', 'Lastname', 'Age', 'Weight'])

print(f'Số dòng sau khi loại trùng lặp : {len(df)}')
print('\n=== DỮ LIỆU SAU KHI LOẠI TRÙNG LẶP ===')
print(df)

Số dòng trước khi loại trùng lặp: 14
Số dòng sau khi loại trùng lặp : 13

=== DỮ LIỆU SAU KHI LOẠI TRÙNG LẶP ===
      Id   Age Weight m0006 m0612 m1218 f0006 f0612 f1218 Firstname Lastname
0    1.0  56.0  70kgs    72    69    71     -     -     -    Mickéy    Mousé
1    2.0  34.0  70kgs     -     -     -    85    84    76    Donald     Duck
2    3.0  16.0    NaN     -     -     -    65    69    72      Mini    Mouse
3    4.0   NaN  78kgs    78    79    72     -     -     -   Scrooge   McDuck
4    5.0  54.0  90kgs     -     -     -    69   NaN    75      Pink  Panther
5    6.0  52.0  85kgs     -     -     -    68    75    72      Huey   McDuck
6    7.0  19.0  56kgs     -     -     -    71    78    75     Dewey   McDuck
7    8.0  32.0  78kgs    78    76    75     -     -     -    Scööpy      Doo
11  10.0  12.0  45kgs     -     -     -    92    95    87     Louie   McDuck
12  11.0   NaN  60kgs    78    75    72     -     -     -     Henry      Nam
13  12.0  34.0    NaN    65    67    55 

---
## Vấn đề 6: Ký tự non-ASCII
Loại bỏ các ký tự không thuộc bảng mã ASCII trong cột Firstname, Lastname

In [13]:
# Xóa các ký tự non-ASCII (mã > 0x7F) khỏi Firstname và Lastname
df['Firstname'] = df['Firstname'].replace({r'[^\x00-\x7F]+': ''}, regex=True)
df['Lastname']  = df['Lastname'].replace({r'[^\x00-\x7F]+': ''}, regex=True)

print('=== DỮ LIỆU SAU KHI LOẠI BỎ KÝ TỰ NON-ASCII ===')
print(df)

=== DỮ LIỆU SAU KHI LOẠI BỎ KÝ TỰ NON-ASCII ===
      Id   Age Weight m0006 m0612 m1218 f0006 f0612 f1218 Firstname Lastname
0    1.0  56.0  70kgs    72    69    71     -     -     -     Micky     Mous
1    2.0  34.0  70kgs     -     -     -    85    84    76    Donald     Duck
2    3.0  16.0    NaN     -     -     -    65    69    72      Mini    Mouse
3    4.0   NaN  78kgs    78    79    72     -     -     -   Scrooge   McDuck
4    5.0  54.0  90kgs     -     -     -    69   NaN    75      Pink  Panther
5    6.0  52.0  85kgs     -     -     -    68    75    72      Huey   McDuck
6    7.0  19.0  56kgs     -     -     -    71    78    75     Dewey   McDuck
7    8.0  32.0  78kgs    78    76    75     -     -     -      Scpy      Doo
11  10.0  12.0  45kgs     -     -     -    92    95    87     Louie   McDuck
12  11.0   NaN  60kgs    78    75    72     -     -     -     Henry      Nam
13  12.0  34.0    NaN    65    67    55     -     -     -    Michel     Long
14  13.0   NaN    NaN     - 

---
## Vấn đề 7: Giá trị bị mất (Missing values) ở Age và Weight

**Yêu cầu:**
- Thống kê dữ liệu thiếu trên từng biến Age và Weight
- Nếu dòng nào có Age **hoặc** Weight có dữ liệu → giữ lại, điền giá trị thiếu bằng **mean của cột Age**
- Nếu thiếu **cả 2** thông tin → xóa dòng

In [14]:
# Thống kê dữ liệu thiếu trên Age và Weight
print('=== THỐNG KÊ DỮ LIỆU THIẾU (Age, Weight) ===')
print(df[['Age', 'Weight']].isnull().sum())
print(f'\nTỉ lệ thiếu Age   : {df["Age"].isnull().mean():.1%}')
print(f'Tỉ lệ thiếu Weight: {df["Weight"].isnull().mean():.1%}')

=== THỐNG KÊ DỮ LIỆU THIẾU (Age, Weight) ===
Age       3
Weight    3
dtype: int64

Tỉ lệ thiếu Age   : 23.1%
Tỉ lệ thiếu Weight: 23.1%


In [15]:
# Bước 1: Xóa dòng nếu CẢ Age và Weight đều thiếu
before = len(df)
df = df.dropna(subset=['Age', 'Weight'], how='all')
after = len(df)
print(f'Đã xóa {before - after} dòng thiếu cả Age và Weight')

# Bước 2: Điền giá trị Age còn thiếu bằng MEAN của cột Age
mean_age = df['Age'].mean()
df['Age'] = df['Age'].fillna(round(mean_age, 1))

print(f'\nMean Age dùng để điền: {mean_age:.2f}')
print('\n=== DỮ LIỆU SAU KHI XỬ LÝ MISSING VALUES (Age, Weight) ===')
print(df)

print('\n=== KIỂM TRA LẠI DỮ LIỆU THIẾU ===')
print(df[['Age', 'Weight']].isnull().sum())

Đã xóa 1 dòng thiếu cả Age và Weight

Mean Age dùng để điền: 36.10

=== DỮ LIỆU SAU KHI XỬ LÝ MISSING VALUES (Age, Weight) ===
      Id   Age Weight m0006 m0612 m1218 f0006 f0612 f1218 Firstname Lastname
0    1.0  56.0  70kgs    72    69    71     -     -     -     Micky     Mous
1    2.0  34.0  70kgs     -     -     -    85    84    76    Donald     Duck
2    3.0  16.0    NaN     -     -     -    65    69    72      Mini    Mouse
3    4.0  36.1  78kgs    78    79    72     -     -     -   Scrooge   McDuck
4    5.0  54.0  90kgs     -     -     -    69   NaN    75      Pink  Panther
5    6.0  52.0  85kgs     -     -     -    68    75    72      Huey   McDuck
6    7.0  19.0  56kgs     -     -     -    71    78    75     Dewey   McDuck
7    8.0  32.0  78kgs    78    76    75     -     -     -      Scpy      Doo
11  10.0  12.0  45kgs     -     -     -    92    95    87     Louie   McDuck
12  11.0  36.1  60kgs    78    75    72     -     -     -     Henry      Nam
13  12.0  34.0    NaN    6

---
## Vấn đề 8: Tiêu đề cột là giá trị, không phải tên biến

Cột như `m0006`, `f1218`,... chứa thông tin: `m`/`f` (giới tính), `0006`/`0612`/`1218` (khoảng giờ),
và giá trị bên trong là **PulseRate** (huyết áp).

→ Cần tách thành 3 cột: `PulseRate`, `Sex`, `Time`

In [16]:
# Melt (unpivot): gộp các cột Sex+Time thành 1 cột duy nhất
df_melted = pd.melt(
    df,
    id_vars=['Id', 'Age', 'Weight', 'Firstname', 'Lastname'],
    value_name='PulseRate',
    var_name='sex_and_time'
).sort_values(['Id', 'Age', 'Weight', 'Firstname', 'Lastname'])

print('=== SAU KHI MELT (UNPIVOT) ===')
print(df_melted.head(10))

=== SAU KHI MELT (UNPIVOT) ===
     Id   Age Weight Firstname Lastname sex_and_time PulseRate
0   1.0  56.0  70kgs     Micky     Mous        m0006        72
12  1.0  56.0  70kgs     Micky     Mous        m0612        69
24  1.0  56.0  70kgs     Micky     Mous        m1218        71
36  1.0  56.0  70kgs     Micky     Mous        f0006         -
48  1.0  56.0  70kgs     Micky     Mous        f0612         -
60  1.0  56.0  70kgs     Micky     Mous        f1218         -
1   2.0  34.0  70kgs    Donald     Duck        m0006         -
13  2.0  34.0  70kgs    Donald     Duck        m0612         -
25  2.0  34.0  70kgs    Donald     Duck        m1218         -
37  2.0  34.0  70kgs    Donald     Duck        f0006        85


In [17]:
# Trích xuất Sex, hours_lower, hours_upper từ cột sex_and_time
# Ví dụ: 'm0006' → Sex='m', hours_lower='00', hours_upper='06'
tmp_df = df_melted['sex_and_time'].str.extract(r'(\D)(\d+)(\d{2})', expand=True)

# Đặt tên cột
tmp_df.columns = ['Sex', 'hours_lower', 'hours_upper']

# Tạo cột Time dựa trên hours_lower và hours_upper
tmp_df['Time'] = tmp_df['hours_lower'] + '-' + tmp_df['hours_upper']

print('=== KẾT QUẢ TRÍCH XUẤT Sex / Time ===')
print(tmp_df.head(10))

=== KẾT QUẢ TRÍCH XUẤT Sex / Time ===
   Sex hours_lower hours_upper   Time
0    m          00          06  00-06
12   m          06          12  06-12
24   m          12          18  12-18
36   f          00          06  00-06
48   f          06          12  06-12
60   f          12          18  12-18
1    m          00          06  00-06
13   m          06          12  06-12
25   m          12          18  12-18
37   f          00          06  00-06


In [18]:
# Gộp (concat) dataframe gốc với dataframe vừa trích xuất
df_final = pd.concat([df_melted.reset_index(drop=True),
                       tmp_df.reset_index(drop=True)], axis=1)

# Xóa các cột không cần thiết và các dòng có giá trị PulseRate bị thiếu ('-')
df_final = df_final.drop(['sex_and_time', 'hours_lower', 'hours_upper'], axis=1)

# Loại bỏ các dòng PulseRate = '-' (không có dữ liệu thật) hoặc NaN
df_final = df_final[df_final['PulseRate'] != '-']
df_final = df_final.dropna(subset=['PulseRate'])

print(f'Số dòng sau khi tách Sex/Time: {len(df_final)}')
print('\n=== KẾT QUẢ CUỐI CÙNG (Vấn đề 8) ===')
print(df_final.head(20))

Số dòng sau khi tách Sex/Time: 35

=== KẾT QUẢ CUỐI CÙNG (Vấn đề 8) ===
     Id   Age Weight Firstname Lastname PulseRate Sex   Time
0   1.0  56.0  70kgs     Micky     Mous        72   m  00-06
1   1.0  56.0  70kgs     Micky     Mous        69   m  06-12
2   1.0  56.0  70kgs     Micky     Mous        71   m  12-18
9   2.0  34.0  70kgs    Donald     Duck        85   f  00-06
10  2.0  34.0  70kgs    Donald     Duck        84   f  06-12
11  2.0  34.0  70kgs    Donald     Duck        76   f  12-18
15  3.0  16.0    NaN      Mini    Mouse        65   f  00-06
16  3.0  16.0    NaN      Mini    Mouse        69   f  06-12
17  3.0  16.0    NaN      Mini    Mouse        72   f  12-18
18  4.0  36.1  78kgs   Scrooge   McDuck        78   m  00-06
19  4.0  36.1  78kgs   Scrooge   McDuck        79   m  06-12
20  4.0  36.1  78kgs   Scrooge   McDuck        72   m  12-18
27  5.0  54.0  90kgs      Pink  Panther        69   f  00-06
29  5.0  54.0  90kgs      Pink  Panther        75   f  12-18
33  6.0  52.0

---
## Yêu cầu thêm: Khảo sát & xử lý tỉ lệ dữ liệu thiếu trên biến huyết áp (PulseRate)

**Thứ tự ưu tiên thay thế giá trị PulseRate bị thiếu của từng người:**
1. Trung bình giá trị liền trước và liền sau của người đó
2. Trung bình 2 giá trị liền trước của người đó
3. Trung bình 2 giá trị liền sau của người đó
4. Trung bình tất cả giá trị huyết áp của người đó
5. Trung bình giá trị huyết áp của nhóm giới tính
6. Trung bình của toàn bộ dữ liệu (hoặc mức ổn định trong y học nếu vẫn không được)

In [19]:
# Chuyển PulseRate sang dạng số (numeric)
df_final['PulseRate'] = pd.to_numeric(df_final['PulseRate'], errors='coerce')

# Khảo sát tỉ lệ dữ liệu thiếu trên biến PulseRate
missing_pulse = df_final['PulseRate'].isnull().sum()
total_pulse   = len(df_final)
print('=== TỈ LỆ DỮ LIỆU THIẾU TRÊN BIẾN PulseRate ===')
print(f'  Số lượng thiếu: {missing_pulse}/{total_pulse} ({missing_pulse/total_pulse:.1%})')

=== TỈ LỆ DỮ LIỆU THIẾU TRÊN BIẾN PulseRate ===
  Số lượng thiếu: 0/35 (0.0%)


In [20]:
def fill_pulse_rate(df_input):
    """
    Xử lý dữ liệu thiếu trên PulseRate theo thứ tự ưu tiên 6 bước.
    """
    df_out = df_input.copy().sort_values(['Id', 'Time']).reset_index(drop=True)

    # Mức ổn định trong y học (giá trị huyết áp trung bình tham khảo)
    MEDICAL_STABLE_VALUE = 75

    overall_mean = df_out['PulseRate'].mean()
    gender_mean  = df_out.groupby('Sex')['PulseRate'].mean()

    for patient_id in df_out['Id'].unique():
        mask = df_out['Id'] == patient_id
        patient_data = df_out.loc[mask, 'PulseRate']

        for idx in patient_data[patient_data.isnull()].index:
            pos = df_out.index.get_loc(idx)
            patient_indices = df_out[mask].index.tolist()
            local_pos = patient_indices.index(idx)

            value = np.nan

            # 1) Trung bình giá trị liền trước và liền sau
            if 0 < local_pos < len(patient_indices) - 1:
                prev_v = df_out.loc[patient_indices[local_pos - 1], 'PulseRate']
                next_v = df_out.loc[patient_indices[local_pos + 1], 'PulseRate']
                if pd.notnull(prev_v) and pd.notnull(next_v):
                    value = (prev_v + next_v) / 2

            # 2) Trung bình 2 giá trị liền trước
            if pd.isnull(value) and local_pos >= 2:
                v1 = df_out.loc[patient_indices[local_pos - 1], 'PulseRate']
                v2 = df_out.loc[patient_indices[local_pos - 2], 'PulseRate']
                if pd.notnull(v1) and pd.notnull(v2):
                    value = (v1 + v2) / 2

            # 3) Trung bình 2 giá trị liền sau
            if pd.isnull(value) and local_pos <= len(patient_indices) - 3:
                v1 = df_out.loc[patient_indices[local_pos + 1], 'PulseRate']
                v2 = df_out.loc[patient_indices[local_pos + 2], 'PulseRate']
                if pd.notnull(v1) and pd.notnull(v2):
                    value = (v1 + v2) / 2

            # 4) Trung bình tất cả giá trị huyết áp của người đó
            if pd.isnull(value):
                patient_mean = patient_data.mean()
                if pd.notnull(patient_mean):
                    value = patient_mean

            # 5) Trung bình theo nhóm giới tính
            if pd.isnull(value):
                sex_val = df_out.loc[idx, 'Sex']
                if sex_val in gender_mean.index and pd.notnull(gender_mean[sex_val]):
                    value = gender_mean[sex_val]

            # 6) Trung bình toàn bộ dữ liệu / mức ổn định y học
            if pd.isnull(value):
                value = overall_mean if pd.notnull(overall_mean) else MEDICAL_STABLE_VALUE

            df_out.loc[idx, 'PulseRate'] = round(value, 1)

    return df_out

df_final = fill_pulse_rate(df_final)

print('=== KIỂM TRA SAU KHI XỬ LÝ PulseRate ===')
print(f'Số giá trị PulseRate còn thiếu: {df_final["PulseRate"].isnull().sum()}')
print('\n=== DỮ LIỆU SAU KHI ĐIỀN PulseRate ===')
print(df_final.head(15))

=== KIỂM TRA SAU KHI XỬ LÝ PulseRate ===
Số giá trị PulseRate còn thiếu: 0

=== DỮ LIỆU SAU KHI ĐIỀN PulseRate ===
     Id   Age Weight Firstname Lastname  PulseRate Sex   Time
0   1.0  56.0  70kgs     Micky     Mous         72   m  00-06
1   1.0  56.0  70kgs     Micky     Mous         69   m  06-12
2   1.0  56.0  70kgs     Micky     Mous         71   m  12-18
3   2.0  34.0  70kgs    Donald     Duck         85   f  00-06
4   2.0  34.0  70kgs    Donald     Duck         84   f  06-12
5   2.0  34.0  70kgs    Donald     Duck         76   f  12-18
6   3.0  16.0    NaN      Mini    Mouse         65   f  00-06
7   3.0  16.0    NaN      Mini    Mouse         69   f  06-12
8   3.0  16.0    NaN      Mini    Mouse         72   f  12-18
9   4.0  36.1  78kgs   Scrooge   McDuck         78   m  00-06
10  4.0  36.1  78kgs   Scrooge   McDuck         79   m  06-12
11  4.0  36.1  78kgs   Scrooge   McDuck         72   m  12-18
12  5.0  54.0  90kgs      Pink  Panther         69   f  00-06
13  5.0  54.0  90

---
## Bước cuối: Rút gọn dữ liệu, reindex và lưu file kết quả

In [21]:
# Sắp xếp lại cột theo thứ tự hợp lý
df_clean = df_final[['Id', 'Firstname', 'Lastname', 'Age', 'Weight',
                      'Sex', 'Time', 'PulseRate']]

# Reindex lại dữ liệu (đánh số index từ 0)
df_clean = df_clean.sort_values(['Id', 'Sex', 'Time']).reset_index(drop=True)

print('=== DỮ LIỆU CUỐI CÙNG SAU KHI LÀM SẠCH ===')
print(f'Shape: {df_clean.shape}')
print(df_clean)

=== DỮ LIỆU CUỐI CÙNG SAU KHI LÀM SẠCH ===
Shape: (35, 8)
      Id Firstname Lastname   Age Weight Sex   Time  PulseRate
0    1.0     Micky     Mous  56.0  70kgs   m  00-06         72
1    1.0     Micky     Mous  56.0  70kgs   m  06-12         69
2    1.0     Micky     Mous  56.0  70kgs   m  12-18         71
3    2.0    Donald     Duck  34.0  70kgs   f  00-06         85
4    2.0    Donald     Duck  34.0  70kgs   f  06-12         84
5    2.0    Donald     Duck  34.0  70kgs   f  12-18         76
6    3.0      Mini    Mouse  16.0    NaN   f  00-06         65
7    3.0      Mini    Mouse  16.0    NaN   f  06-12         69
8    3.0      Mini    Mouse  16.0    NaN   f  12-18         72
9    4.0   Scrooge   McDuck  36.1  78kgs   m  00-06         78
10   4.0   Scrooge   McDuck  36.1  78kgs   m  06-12         79
11   4.0   Scrooge   McDuck  36.1  78kgs   m  12-18         72
12   5.0      Pink  Panther  54.0  90kgs   f  00-06         69
13   5.0      Pink  Panther  54.0  90kgs   f  12-18         

In [22]:
# Lưu trữ dữ liệu đã xử lý thành công
df_clean.to_csv('patient_heart_rate_clean.csv', index=False)
print('✅ Đã lưu file: patient_heart_rate_clean.csv')

# Download file về máy
from google.colab import files
files.download('patient_heart_rate_clean.csv')

✅ Đã lưu file: patient_heart_rate_clean.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>